In [1]:
import os, torch, gc, nltk
import numpy as np
import pandas as pd
from nltk.corpus import reuters
from sentence_transformers import SentenceTransformer

In [2]:
output_dir = "src/data/rcv1_v2"
os.makedirs(output_dir, exist_ok=True)
nltk.download('reuters')
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

[nltk_data] Downloading package reuters to /home/raosidha/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


In [3]:
all_ids = reuters.fileids()
deep_entries = []
MAX_DOCS = 2500 # Start with 2500 to ensure stability; move to 5000 if it passes

print(f"Scanning Reuters for depth >= 2...")
for fid in all_ids:
    cats = list(set(reuters.categories(fid)))
    if len(cats) >= 2:
        # TRUNCATE TEXT: Keeping the first 1000 chars prevents OOM crashes
        text = reuters.raw(fid).replace('\n', ' ')[:1000]
        deep_entries.append({
            "fileid": fid,
            "topic": text,
            "category 0": cats[0],
            "category 1": cats[1]
            #"category 2": cats[2]
        })
    if len(deep_entries) >= MAX_DOCS: break

df = pd.DataFrame(deep_entries)

Scanning Reuters for depth >= 2...


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

print(f"Embedding {len(df)} docs on {device}...")
embeddings = model.encode(
    df["topic"].tolist(),
    batch_size=2, # Tiny batch size = Maximum stability
    show_progress_bar=True,
    convert_to_numpy=True
)

Embedding 1628 docs on cpu...


Batches:   0%|          | 0/814 [00:00<?, ?it/s]

In [5]:
np.save(os.path.join(output_dir, "rcv1_qwen_embeddings.npy"), embeddings)
df.to_csv(os.path.join(output_dir, "rcv1_qwen_metadata.csv"), index=False)
print("Phase 1 Complete: Deep Data Saved.")

Phase 1 Complete: Deep Data Saved.
